# 04. Differential expression with DESeq2

このノートブックは、gene count matrixからDEGを検出する段階である。

**このノートブックの役割**

- count matrix、sample metadata、contrast表の対応を確認する。
- `scripts/deseq2_differential_expression.R` を実行し、指定したcontrastごとにDESeq2の結果を作る。
- 後続の可視化と解釈に使うnormalized counts、VST counts、MA plotを出力する。

**入力**

- `results/counts/gene_counts.tsv`（行=gene、列=sampleのraw count matrix）
- `metadata/samples.tsv`（sample_id、condition、replicateを持つsample metadata）
- `metadata/contrasts.tsv`（test conditionとreference conditionの比較表）

**出力**

- `results/de/deseq2_<contrast_id>.csv`（log2FoldChange、p-value、padjなどを含むDESeq2結果）
- `results/de/deseq2_normalized_counts.tsv`（size factor補正後のcount）
- `results/de/deseq2_vst_counts.tsv`（PCAやheatmap向けのVST変換値）
- `results/de/MA_<contrast_id>.pdf`

**次に進む条件**

- 各contrastのDESeq2結果CSVが作られている。
- log2 fold changeの向きが `contrasts.tsv` のtest/referenceと一致している。


## DESeq2モデルと統計的検定の背景

DESeq2は、gene $g$、sample $s$ のread count（リード数）を**負の二項分布（Negative Binomial; NB distribution）**でモデル化する。

$$Y_{g,s} \sim \mathrm{NB}(\mu_{g,s}, \alpha_g)$$

ここで、$\mu_{g,s}$ は平均count、$\alpha_g$ は過分散（dispersion）パラメータである。
生物学的反復サンプルのばらつきは、ポアソン分布（平均＝分散）では説明できない過分散を持つため、分散 $\mathrm{Var}(Y_{g,s})$ は以下のように定式化される。

$$\mathrm{Var}(Y_{g,s}) = \mu_{g,s} + \alpha_g \mu_{g,s}^2$$

### $p$-value とは何か
統計的検定における $p$-value（$p$値）とは、「帰無仮説 $H_0$（群間で発現量に差がない：$\log_2 \mathrm{FC} = 0$）が正しいと仮定したとき、得られたデータと同等かそれ以上に極端な結果が得られる確率」である。
$p$ が極めて小さい（通常 0.05 未満）場合、帰無仮説は棄却され、「統計的に有意な差がある」と判断する。

### $padj$（多重検定補正後の $p$値）とは何か
数万個のgeneを同時に検定する**多重検定（multiple testing）**では、各geneを単に $p < 0.05$ で判定すると、偶然だけで有意と判定される**偽陽性（False Positive）**が大量に発生する（例えば 20,000 geneに対して検定すると、差がなくても約 1,000 個が偶然 $p < 0.05$ になる）。
この問題を防ぐため、DESeq2では **Benjamini-Hochberg（BH）法** を用いて $p$-value を補正し、**FDR（False Discovery Rate）：有意と判定されたgene群の中に含まれる偽陽性の割合** を一定値以下に制御する。
FDR補正後の $p$値は $padj$（adjusted $p$-value）と表記され、解析の実務ではこの $padj < 0.05$ を閾値として有意な DEG（Differentially Expressed Gene）を特定する。

$$P_{(i)} \le \frac{i}{m} Q$$

ここで $P_{(i)}$ は順位 $i$ 番目の $p$値、$m$ は総検定数（gene数）、$Q$ は目標とする FDR（False Discovery Rate）閾値（例えば 0.05）を指す。


In [1]:
from pathlib import Path
import json
import shutil
import subprocess

import pandas as pd

PROJECT_DIR = Path("/Users/yusuke_tateishi/Documents/RNA_seq").resolve()
CONFIG = json.loads((PROJECT_DIR / "config" / "analysis_config.json").read_text(encoding="utf-8"))
COUNTS_PATH = PROJECT_DIR / CONFIG["differential_expression"]["count_matrix_path"]
SAMPLES_PATH = PROJECT_DIR / CONFIG["samples_path"]
CONTRASTS_PATH = PROJECT_DIR / CONFIG["contrasts_path"]
DE_DIR = PROJECT_DIR / "results" / "de"
DE_DIR.mkdir(parents=True, exist_ok=True)

for path in [COUNTS_PATH, SAMPLES_PATH, CONTRASTS_PATH]:
    print(path.relative_to(PROJECT_DIR), "exists:", path.exists())


results/counts/gene_counts.tsv exists: True
metadata/samples.tsv exists: True
metadata/contrasts.tsv exists: True


## 入力表の確認

count matrixのサンプル列とmetadataの `sample_id` が一致している必要がある。


In [2]:
if not COUNTS_PATH.exists():
    raise FileNotFoundError(f"Count matrix not found: {COUNTS_PATH}. Run notebook 02 first.")

counts = pd.read_csv(COUNTS_PATH, sep="\t", nrows=5)
samples = pd.read_csv(SAMPLES_PATH, sep="\t")
contrasts = pd.read_csv(CONTRASTS_PATH, sep="\t")

count_sample_columns = [column for column in counts.columns if column != "gene_id"]
print("count matrix sample columns:", count_sample_columns)
print("metadata sample IDs:", list(samples["sample_id"]))
print("missing in counts:", sorted(set(samples["sample_id"]) - set(count_sample_columns)))
print("extra in counts:", sorted(set(count_sample_columns) - set(samples["sample_id"])))
display(contrasts)


count matrix sample columns: ['NAC_S2_2h_1', 'NAC_S2_2h_2', 'NAC_S2_2h_3', 'Non_1', 'Non_2', 'Non_3', 'Oxi_2h_1', 'Oxi_2h_2', 'Oxi_2h_3']
metadata sample IDs: ['NAC_S2_2h_1', 'NAC_S2_2h_2', 'NAC_S2_2h_3', 'Non_1', 'Non_2', 'Non_3', 'Oxi_2h_1', 'Oxi_2h_2', 'Oxi_2h_3']
missing in counts: []
extra in counts: []


,contrast_id,test_condition,reference_condition,description
0,Oxi_2h_vs_Non,Oxi_2h,Non,Oxidative stress at 2h compared with untreated...
1,NAC_S2_2h_vs_Non,NAC_S2_2h,Non,NAC_S2 at 2h compared with untreated control
2,NAC_S2_2h_vs_Oxi_2h,NAC_S2_2h,Oxi_2h,NAC_S2 at 2h compared with oxidative stress at 2h


## DESeq2実行コマンド

実際のDESeq2処理は `scripts/deseq2_differential_expression.R` に置いている。ノートブックは入力確認と実行管理に集中する。


### `scripts/deseq2_differential_expression.R` コマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `Rscript` | Rスクリプトをコマンドラインから実行するためのコマンド。 |
| `scripts/deseq2_differential_expression.R` | DESeq2解析本体のスクリプト。ノートブックは入力確認と実行管理を担当する。 |
| `--counts=FILE` | DESeq2へ渡すraw count matrix。行=gene、列=sampleである。 |
| `--metadata=FILE` | `sample_id` と `condition` などを持つsample metadata。count matrixの列名と対応している必要がある。 |
| `--contrasts=FILE` | 比較するtest condition/reference conditionを定義した表。 |
| `--outdir=DIR` | DESeq2結果CSV、normalized counts、VST counts、MA plotの出力先。 |
| `--design=FORMULA` | DESeq2のdesign formula。例: `~ condition`。 |
| `--reference=GROUP` | log2 fold changeの基準群。ここを基準にtest conditionの上昇/低下を読む。 |
| `--min-count=N` | 低発現geneを除くための最小count閾値。 |
| `--min-samples=N` | `--min-count` 以上を満たす必要がある最小サンプル数。 |
| `--alpha=X` | adjusted p-value（padj）の有意水準。 |

DESeq2の主な読み方は、`log2FoldChange > 0` がtest condition側で高い、`log2FoldChange < 0` がreference condition側で高い、である。


In [3]:
DESEQ2_SCRIPT = PROJECT_DIR / "scripts" / "deseq2_differential_expression.R"

deseq2_command = [
    "Rscript",
    str(DESEQ2_SCRIPT),
    f"--counts={COUNTS_PATH}",
    f"--metadata={SAMPLES_PATH}",
    f"--contrasts={CONTRASTS_PATH}",
    f"--outdir={DE_DIR}",
    f"--design={CONFIG['differential_expression']['design_formula']}",
    f"--reference={CONFIG['differential_expression']['reference_condition']}",
    f"--min-count={CONFIG['differential_expression']['min_count']}",
    f"--min-samples={CONFIG['differential_expression']['min_samples']}",
    f"--alpha={CONFIG['differential_expression']['alpha']}",
]

command_path = DE_DIR / "deseq2_command.txt"
command_path.write_text(" ".join(deseq2_command) + "\n", encoding="utf-8")
print("Command written to:", command_path.relative_to(PROJECT_DIR))
print(" ".join(deseq2_command))


Command written to: results/de/deseq2_command.txt
Rscript /Users/yusuke_tateishi/Documents/RNA_seq/scripts/deseq2_differential_expression.R --counts=/Users/yusuke_tateishi/Documents/RNA_seq/results/counts/gene_counts.tsv --metadata=/Users/yusuke_tateishi/Documents/RNA_seq/metadata/samples.tsv --contrasts=/Users/yusuke_tateishi/Documents/RNA_seq/metadata/contrasts.tsv --outdir=/Users/yusuke_tateishi/Documents/RNA_seq/results/de --design=~ condition --reference=Non --min-count=10 --min-samples=2 --alpha=0.05


In [4]:
RUN_DESEQ2 = True

if RUN_DESEQ2:
    rscript = shutil.which("Rscript")
    if rscript is None:
        raise RuntimeError("Rscript was not found. Activate the rna-seq environment first.")
    deseq2_command[0] = rscript
    subprocess.run(deseq2_command, check=True)
else:
    print("Set RUN_DESEQ2 = True after sample QC looks acceptable.")


estimating size factors
estimating dispersions
gene-wise dispersion estimates
mean-dispersion relationship
final dispersion estimates
fitting model and testing


## 結果の読み方

DESeq2結果の主な列である。

- `baseMean`: そのgeneの平均的な発現量。
- `log2FoldChange`: test condition / reference condition のlog2比。
- `pvalue`: 統計検定のp値。
- `padj`: 多重検定補正後のp値。通常はこちらを重視する。
- `direction`: このプロジェクトのRスクリプトが付けた簡易ラベル。


In [5]:
result_files = sorted(DE_DIR.glob("deseq2_*.csv"))
result_files = [path for path in result_files if path.name not in {"deseq2_normalized_counts.tsv", "deseq2_vst_counts.tsv"}]

summaries = []
for path in result_files:
    de = pd.read_csv(path)
    summaries.append(
        {
            "file": path.name,
            "rows": len(de),
            "padj_lt_alpha": int((de["padj"] < CONFIG["differential_expression"]["alpha"]).sum()),
            "up": int((de["direction"] == "up").sum()) if "direction" in de.columns else None,
            "down": int((de["direction"] == "down").sum()) if "direction" in de.columns else None,
        }
    )

if summaries:
    display(pd.DataFrame(summaries))
else:
    print("No DESeq2 result files yet. Run DESeq2 first.")


,file,rows,padj_lt_alpha,up,down
0,deseq2_NAC_S2_2h_vs_Non.csv,15811,751,349,402
1,deseq2_NAC_S2_2h_vs_Oxi_2h.csv,15811,1031,416,615
2,deseq2_Oxi_2h_vs_Non.csv,15811,41,28,13


**よくある間違い**

- `padj` ではなく、生の `pvalue` だけで判断する。
- PCAで外れ値があるのに、そのままDESeq2へ進む。
- `reference_condition` を確認せず、log2 fold changeの向きを誤解する。

**小さい練習**

`Oxi_2h_vs_Non` の `log2FoldChange > 0` は「Oxi_2hでNonより高い」という意味である。この向きを自分の言葉で確認しよう。
